# Synthetic broadband targeting

400 generated grid cells demonstrate spatial statistics, regression and population-proxy coverage. They are not authentic Census tracts, measured households or an eligibility list. The saved graph has 45 disconnected components and mean degree 4.25. There is no observed policy effect. See [REPORT.md](../REPORT.md).


## Generated data

A shared latent proximity field drives generated income, education, population and adoption. Smoothing and suppression in 15 urban-labelled cells create structure. Area categories are synthetic. The generator does not measure infrastructure, prices or affordability constraints.


In [1]:
import sys, yaml, json
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

from src.geo_urban.fetch import make_synthetic_geodataframe
from src.geo_urban.spatial_stats import build_weights, global_morans_i, lisa_clusters, subgroup_morans_i
from src.geo_urban.regression import run_ols, run_spatial_lag, run_spatial_error, compare_models, run_subgroup_ols, population_coverage_analysis
from src.geo_urban.maps import choropleth_map, lisa_map, county_type_map
from src.geo_urban.visualise import (
    plot_lisa_static, plot_moran_scatter, plot_regression_comparison,
    plot_broadband_distribution, plot_subgroup_regression,
    plot_guardrail, plot_edi_by_cluster,
)

OUTPUTS = ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)

cfg = yaml.safe_load((ROOT / 'config' / 'default.yaml').read_text())
geo_cfg = cfg['geography']
reg_cfg = cfg['regression']
scenario_cfg = cfg['scenario']

sns.set_theme(style='whitegrid', font_scale=1.05)
print('Config loaded.')
print(f"Scenario: {scenario_cfg['name']}")
print(f"Broadband threshold for 'underserved': {scenario_cfg['broadband_threshold']:.0%}")

Config loaded.
Scenario: California BEAD Broadband Subsidy Targeting
Broadband threshold for 'underserved': 65%


In [ ]:
USE_SYNTHETIC = True   # Set False + provide CENSUS_API_KEY for real ACS data
CENSUS_API_KEY = None

if USE_SYNTHETIC:
    gdf = make_synthetic_geodataframe(
        n_tracts=scenario_cfg['n_tracts_synthetic'],
        seed=scenario_cfg['synthetic_seed'],
    )
    print(f'Loaded {len(gdf):,} synthetic cells.')
else:
    from src.geo_urban.fetch import fetch_acs_data, fetch_tiger_shapefile
    from src.geo_urban.wrangle import merge_acs_tiger
    acs_df = fetch_acs_data(geo_cfg['state_fips'], cfg['acs_variables'],
                            year=geo_cfg['census_year'], census_api_key=CENSUS_API_KEY)
    tiger_gdf = fetch_tiger_shapefile(geo_cfg['state_fips'], year=geo_cfg['census_year'])
    gdf = merge_acs_tiger(acs_df, tiger_gdf,
                          projection_epsg=geo_cfg['projection_epsg'],
                          min_population=reg_cfg['min_population'])
    print(f'Loaded {len(gdf):,} real ACS tracts.')

print(f"\nCounty type breakdown:")
print(gdf['county_type'].value_counts().to_string())
print(f"\nBroadband rate summary:")
print(gdf['broadband_rate'].describe().round(3).to_string())

## Exploratory association

Inspect generated distributions and bivariate correlations. Correlation in a constructed scenario is not a causal mechanism.


In [ ]:
THRESHOLD = scenario_cfg['broadband_threshold']
underserved_count = (gdf['broadband_rate'] < THRESHOLD).sum()
underserved_pct = underserved_count / len(gdf) * 100

print(f"Underserved tracts (broadband < {THRESHOLD:.0%}): {underserved_count} ({underserved_pct:.1f}%)")
print(f"Underserved population proxy: {gdf.loc[gdf['broadband_rate'] < THRESHOLD, 'population'].sum():,}")
print()

# Income-broadband correlation
r_inc_bb = gdf[['median_household_income','broadband_rate']].corr().iloc[0,1]
r_edu_bb = gdf[['bach_rate','broadband_rate']].corr().iloc[0,1]
print(f"Pearson r (income × broadband): {r_inc_bb:.3f}")
print(f"Pearson r (education × broadband): {r_edu_bb:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(gdf['broadband_rate'], bins=30, color='#4472C4', edgecolor='white', alpha=0.8)
axes[0].axvline(THRESHOLD, color='red', linewidth=1.5, linestyle='--', label=f'Threshold ({THRESHOLD:.0%})')
axes[0].set_xlabel('Broadband Rate')
axes[0].set_ylabel('Tract Count')
axes[0].set_title('Distribution of Broadband Rates')
axes[0].legend()

axes[1].scatter(gdf['median_household_income'], gdf['broadband_rate'],
                alpha=0.4, s=12, color='#4472C4')
axes[1].set_xlabel('Median Household Income ($)')
axes[1].set_ylabel('Broadband Rate')
axes[1].set_title(f'Income vs Broadband\nr = {r_inc_bb:.3f}')

axes[2].scatter(gdf['bach_rate'], gdf['broadband_rate'],
                alpha=0.4, s=12, color='#70AD47')
axes[2].set_xlabel('Bachelor\'s Degree Rate')
axes[2].set_ylabel('Broadband Rate')
axes[2].set_title(f'Education vs Broadband\nr = {r_edu_bb:.3f}')

plt.suptitle('Exploratory Analysis, Broadband Access vs Key Predictors', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUTS / 'exploratory_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved exploratory_correlation.png')

In [4]:
# Broadband distribution by county type, reveals the structural heterogeneity
fig = plot_broadband_distribution(gdf, broadband_threshold=THRESHOLD,
                                  save_path=OUTPUTS / 'broadband_distribution.png')
plt.show()
print('Saved broadband_distribution.png')

print('\nMedian broadband rate by county type:')
print(gdf.groupby('county_type')['broadband_rate'].agg(['median','mean','min','max']).round(3).to_string())

Saved broadband_distribution.png

Median broadband rate by county type:
             median   mean    min    max
county_type                             
rural         0.274  0.301  0.180  0.497
suburban      0.598  0.596  0.466  0.704
urban         0.801  0.781  0.363  0.960


## Synthetic choropleths

Maps display generated income, adoption and area types. Counts use a population proxy and do not identify actual California places or service availability.


In [ ]:
print('Building income choropleth ...')
_ = choropleth_map(
    gdf, column='median_household_income',
    title='Median Household Income by Census Tract',
    caption='Median Household Income ($)',
    colormap='YlOrRd',
    save_path=OUTPUTS / 'choropleth_income.html',
)
print('  → choropleth_income.html')

print('Building broadband choropleth ...')
_ = choropleth_map(
    gdf, column='broadband_rate',
    title='Broadband Subscription Rate by Census Tract',
    caption='Synthetic adoption fraction (not infrastructure availability)',
    colormap='Blues',
    save_path=OUTPUTS / 'choropleth_broadband.html',
)
print('  → choropleth_broadband.html')

print('Building county type map ...')
_ = county_type_map(gdf, save_path=OUTPUTS / 'choropleth_county_type.html')
print('  → choropleth_county_type.html')

print('\nAll three HTML maps saved to outputs/. Open in a browser to explore interactively.')

## Global Moran statistic

Saved I=.913804 and pseudo p=.001 from 999 permutations describe constructed association under the saved fragmented graph. A rounded normal p-value of zero is not literal zero probability. Global association does not establish that observed controls fail or that an intervention is justified.


In [ ]:
print('Building spatial weights (Queen contiguity, row-standardised) ...')
W = build_weights(gdf, queen=cfg['spatial']['queen_contiguity'])
print(f'  Cells: {W.n}   Average neighbours: {W.mean_neighbors:.1f}   Min: {W.min_neighbors}   Max: {W.max_neighbors}')

print('\nRunning Global Moran\'s I (999 permutations) ...')
moran_result = global_morans_i(
    gdf, 'broadband_rate', W,
    n_permutations=cfg['spatial']['n_permutations'],
)

print(f"\n  Moran's I        = {moran_result['moran_i']:.4f}")
print(f"  Expected I (H0)  = {moran_result['expected_i']:.4f}")
print(f"  z-score          = {moran_result['z_score']:.2f}")
print(f"  p (normal)       = {moran_result['p_value_norm']:.6f}")
print(f"  p (sim, primary) = {moran_result['p_value_sim']:.6f}")
print()
print("Constructed association only; no causal or policy conclusion. Inspect weights connectivity.")


In [7]:
# We'll add LISA clusters first (needed for colouring the scatter) and come back
# For now, run a quick LISA pass for the scatter plot colouring
gdf = lisa_clusters(
    gdf, 'broadband_rate', W,
    significance=cfg['spatial']['lisa_significance'],
    n_permutations=cfg['spatial']['n_permutations'],
)

fig = plot_moran_scatter(
    gdf, 'broadband_rate', W,
    moran_i=moran_result['moran_i'],
    p_value=moran_result['p_value_sim'],
    save_path=OUTPUTS / 'moran_scatter.png',
)
plt.show()
print('Saved moran_scatter.png')
print('Slope of the Moran scatter (= Moran\'s I): the strong positive gradient confirms clustering.')
print('Points are coloured by their subsequent LISA classification.')

Saved moran_scatter.png
Slope of the Moran scatter (= Moran's I): the strong positive gradient confirms clustering.
Points are coloured by their subsequent LISA classification.


## Local exploratory labels

113 LL, 92 HH, four LH and 191 NS labels use uncorrected local pseudo p<.05. All LL cells have the rural label. NS does not mean adequately served. The local permutation seed was not recorded. Relative clustering does not diagnose market failure, affordability or grant eligibility.


In [ ]:
# LISA already computed in cell 4b. Summarise results.
cluster_counts = gdf['lisa_cluster'].value_counts()
print('LISA cluster counts (broadband_rate):')
print(cluster_counts.to_string())
print()

ll_count = cluster_counts.get('LL', 0)
hh_count = cluster_counts.get('HH', 0)
ll_pct = 100 * ll_count / len(gdf)

print(f"  Cold spots (LL): {ll_count} tracts ({ll_pct:.1f}%), relative low-low labels, not eligibility")
print(f"  Hot spots  (HH): {hh_count} tracts ({100*hh_count/len(gdf):.1f}%), relative high-high labels, not service availability")

print('\nLL cold spots by county type:')
ll_by_type = gdf[gdf['lisa_cluster']=='LL']['county_type'].value_counts()
print(ll_by_type.to_string())

print('\nMedian population per tract by LISA cluster:')
print(gdf.groupby('lisa_cluster')['population'].median().round(0).to_string())

In [9]:
fig = plot_lisa_static(gdf, save_path=OUTPUTS / 'lisa_cluster_map.png')
plt.show()

_ = lisa_map(gdf, save_path=OUTPUTS / 'choropleth_lisa.html')
print('Saved lisa_cluster_map.png and choropleth_lisa.html')

Saved lisa_cluster_map.png and choropleth_lisa.html


d:\Users\jesco\Documents\Github\geospatial-urban-analytics\src\geo_urban\maps.py:90: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centre = [gdf_wgs84.geometry.centroid.y.mean(), gdf_wgs84.geometry.centroid.x.mean()]


## Generated disadvantage index

EDI combines generated income, education and broadband. Its broadband component uses values before urban suppression. It overlaps the outcome and cannot independently confirm need or policy effectiveness.


In [10]:
print('EDI summary statistics:')
print(gdf['economic_disadvantage_index'].describe().round(3).to_string())

print('\nMedian EDI by LISA cluster:')
print(gdf.groupby('lisa_cluster')['economic_disadvantage_index'].median().round(3).sort_values(ascending=False).to_string())

print('\nMedian EDI by county type:')
print(gdf.groupby('county_type')['economic_disadvantage_index'].median().round(3).sort_values(ascending=False).to_string())

fig = plot_edi_by_cluster(gdf, save_path=OUTPUTS / 'edi_by_cluster.png')
plt.show()
print('Saved edi_by_cluster.png')

d:\Users\jesco\Documents\Github\geospatial-urban-analytics\src\geo_urban\visualise.py:315: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(


EDI summary statistics:
count    400.000
mean       0.613
std        0.192
min        0.156
25%        0.471
50%        0.604
75%        0.789
max        0.973

Median EDI by LISA cluster:
lisa_cluster
LL    0.840
NS    0.583
LH    0.513
HH    0.385

Median EDI by county type:
county_type
rural       0.818
suburban    0.574
urban       0.409


Saved edi_by_cluster.png


## Generated area-type comparisons

Subgroup weights change with each subset and have islands. Association and fit summaries describe designed groups, not identified urban or rural mechanisms.


In [11]:
print('Running subgroup Moran\'s I (each county type gets its own weights matrix) ...')
sub_moran = subgroup_morans_i(
    gdf, 'broadband_rate',
    group_col='county_type',
    n_permutations=cfg['spatial']['n_permutations'],
)

print('\nMoran\'s I by county type:')
display_cols = ['group','n','moran_i','z_score','p_value_sim']
print(sub_moran[display_cols].to_string(index=False))

Running subgroup Moran's I (each county type gets its own weights matrix) ...
('WARNING: ', 152, ' is an island (no neighbors)')
('WARNING: ', 33, ' is an island (no neighbors)')
('WARNING: ', 41, ' is an island (no neighbors)')
('WARNING: ', 20, ' is an island (no neighbors)')
('WARNING: ', 58, ' is an island (no neighbors)')
('WARNING: ', 76, ' is an island (no neighbors)')

Moran's I by county type:
   group   n  moran_i  z_score  p_value_sim
   rural 161 0.916006  14.0327        0.001
suburban 107 0.591550   6.7642        0.001
   urban 132 0.325309   5.1387        0.001


d:\Users\jesco\Documents\Github\geospatial-urban-analytics\src\geo_urban\spatial_stats.py:14: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  W = Queen.from_dataframe(gdf) if queen else Rook.from_dataframe(gdf)
d:\Users\jesco\Documents\Github\geospatial-urban-analytics\.venv\Lib\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 24 disconnected components.
 There is 1 island with id: 152.
  W.__init__(self, neighbors, ids=ids, **kw)
d:\Users\jesco\Documents\Github\geospatial-urban-analytics\src\geo_urban\spatial_stats.py:14: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  W = Queen.from_dataframe(gdf) if queen else Rook.from_dataframe(gdf)
d:\Users\jesco\Documents\Github\geospatial-urban-analytics\.venv\Li

In [12]:
print('Running subgroup OLS ...')
dep = reg_cfg['dependent_variable']
covariates = [c for c in reg_cfg['covariates'] if c in gdf.columns]

subgroup_ols_df = run_subgroup_ols(gdf, dep, covariates)

print('\nOLS R² by county type:')
summary_cols = ['group','n','r_squared','adj_r_squared','aic']
summary = subgroup_ols_df.drop_duplicates('group')[summary_cols]
print(summary.to_string(index=False))

print('\nlog_income coefficient by county type:')
inc_coef = subgroup_ols_df[subgroup_ols_df['variable']=='log_income'][['group','coef','p_value']]
print(inc_coef.to_string(index=False))

fig = plot_subgroup_regression(subgroup_ols_df, save_path=OUTPUTS / 'subgroup_regression.png')
plt.show()
print('Saved subgroup_regression.png')

Running subgroup OLS ...

OLS R² by county type:
   group   n  r_squared  adj_r_squared     aic
   rural 161     0.4290         0.4181 -380.91
suburban 107     0.3013         0.2809 -318.55
   urban 132     0.3226         0.3067 -206.03

log_income coefficient by county type:
   group     coef  p_value
   rural 0.058673 0.002259
suburban 0.060428 0.000081
   urban 0.206418 0.000000


Saved subgroup_regression.png


## In-sample spatial model comparison

OLS R2=.7549 and Spatial Lag pseudo R2=.9106 are distinct measures. Spatial pseudo R2 is squared fitted-value correlation. AIC -922.24 versus -591.90 supports only an in-sample comparison. Rho .677804 and lambda .851843 are fitted dependence parameters, not intervention effects.


In [ ]:
print(f'Dependent variable: {dep}')
print(f'Covariates: {covariates}')
print()

ols_result = run_ols(gdf, dep, covariates)
print(f"OLS:            R² = {ols_result['r_squared']:.4f}   AIC = {ols_result['aic']:.1f}")
print('  Coefficients:')
for var, stats in ols_result['coefficients'].items():
    stars = '***' if stats['p_value'] < 0.001 else '**' if stats['p_value'] < 0.01 else '*' if stats['p_value'] < 0.05 else '  '
    print(f"    {var:20s}: coef = {stats['coef']:+.5f}   p = {stats['p_value']:.5f} {stars}")

lag_result = run_spatial_lag(gdf, dep, covariates, W)
if 'error' not in lag_result:
    print(f"\nSpatial Lag:    pseudo R² = {lag_result['r_squared']:.4f}   AIC = {lag_result['aic']:.1f}   ρ = {lag_result['rho']:.4f}")

err_result = run_spatial_error(gdf, dep, covariates, W)
if 'error' not in err_result:
    print(f"Spatial Error:  pseudo R² = {err_result['r_squared']:.4f}   AIC = {err_result['aic']:.1f}   λ = {err_result['lambda']:.4f}")

comparison = compare_models([ols_result, lag_result, err_result])
print('\nModel comparison table:')
print(comparison.to_string(index=False))

In [ ]:
comparison.to_csv(OUTPUTS / 'regression_comparison.csv', index=False)

fig = plot_regression_comparison(comparison, save_path=OUTPUTS / 'regression_comparison.png')
plt.show()
print('Saved regression_comparison.csv and regression_comparison.png')

best_model = comparison.loc[comparison['AIC'].idxmin(), 'Method']
aic_improvement = ols_result['aic'] - comparison['AIC'].min()
print(f"\nBest model by AIC: {best_model}")
print(f"AIC improvement over OLS: {aic_improvement:.1f} points")
print("In-sample model fit does not establish intervention effects.")


## Population-proxy coverage

Each cell below 65% adoption contributes its entire population proxy; this is not population multiplied by non-adoption. LL includes 119,165/521,152 units, 22.9%. Outside-target units are 99,455 rural, 234,282 suburban and 68,250 urban, total 401,987 or 77.1%. The threshold is a scenario setting. Neither rule establishes which public intervention works.


In [ ]:
coverage = population_coverage_analysis(gdf, broadband_threshold=THRESHOLD)

print('=== Guardrail: Population Coverage Analysis ===')
print(f"Broadband threshold: {coverage['broadband_threshold']:.0%}")
print(f"Total below-threshold synthetic cells: {coverage['total_underserved_tracts']}")
print(f"Total underserved proxy units: {coverage['total_underserved_hh']:,}")
print()
print(f"LL synthetic cells: {coverage['ll_cluster_tracts']}")
print(f"LL tracts that are also underserved: {coverage['ll_underserved_tracts']}")
print(f"Underserved proxy units in LL tracts: {coverage['ll_underserved_hh']:,} ({coverage['ll_coverage_pct']:.1f}%)")
print()
print(f"MISSED by LL targeting: {coverage['missed_hh']:,} proxy units ({coverage['missed_pct']:.1f}%)")
print()

print("Targeting coverage is arithmetic on synthetic proxy units, not policy effectiveness.")

print('\nCoverage by synthetic area type:')
for ct, data in coverage['coverage_by_type'].items():
    print(f"  {ct:10s}: {data['underserved_tracts']} underserved tracts, "
          f"{data['underserved_hh']:,} proxy units | LL covers {data['ll_coverage_pct']:.0f}% of proxy units")

In [16]:
fig = plot_guardrail(gdf, coverage, broadband_threshold=THRESHOLD,
                     save_path=OUTPUTS / 'guardrail_analysis.png')
plt.show()
print('Saved guardrail_analysis.png')

Saved guardrail_analysis.png


## Synthesis and export

The workflow demonstrates association, model fitting and exact targeting arithmetic on artificial cells. The explorer faithfully preserves its saved labels and proxy denominator. Repairing topology requires a new run with a recorded graph/seeds and synchronized outputs; authentic geography and an evaluation design are needed for empirical policy questions.


In [17]:
run_summary = {
    'scenario': scenario_cfg['name'],
    'n_tracts': len(gdf),
    'data_source': 'synthetic (structured latent model)' if USE_SYNTHETIC else 'ACS 5-year 2022, California',
    'broadband_threshold': THRESHOLD,

    'exploratory': {
        'r_income_broadband': round(float(r_inc_bb), 4),
        'r_education_broadband': round(float(r_edu_bb), 4),
        'underserved_tracts': int(underserved_count),
        'underserved_pct': round(float(underserved_pct), 2),
        'broadband_by_county_type': gdf.groupby('county_type')['broadband_rate'].median().round(3).to_dict(),
    },

    'global_moran': moran_result,

    'subgroup_moran': sub_moran[['group','n','moran_i','p_value_sim']].to_dict(orient='records'),

    'lisa_cluster_counts': cluster_counts.to_dict(),
    'lisa_by_county_type': gdf.groupby(['county_type','lisa_cluster']).size().unstack(fill_value=0).to_dict(),

    'edi_by_cluster': gdf.groupby('lisa_cluster')['economic_disadvantage_index'].median().round(4).to_dict(),
    'edi_by_county_type': gdf.groupby('county_type')['economic_disadvantage_index'].median().round(4).to_dict(),

    'ols_full': {k: v for k, v in ols_result.items() if k != 'coefficients'},
    'ols_coefficients': ols_result['coefficients'],
    'spatial_lag': lag_result,
    'spatial_error': err_result,

    'subgroup_ols_r2': subgroup_ols_df.drop_duplicates('group').set_index('group')['r_squared'].round(4).to_dict(),
    'subgroup_ols_income_coef': subgroup_ols_df[subgroup_ols_df['variable']=='log_income'].set_index('group')[['coef','p_value']].round(5).to_dict(orient='index'),

    'guardrail': {
        'broadband_threshold': coverage['broadband_threshold'],
        'total_underserved_tracts': coverage['total_underserved_tracts'],
        'total_underserved_hh': coverage['total_underserved_hh'],
        'll_cluster_tracts': coverage['ll_cluster_tracts'],
        'll_underserved_hh': coverage['ll_underserved_hh'],
        'll_coverage_pct': coverage['ll_coverage_pct'],
        'missed_hh': coverage['missed_hh'],
        'missed_pct': coverage['missed_pct'],
        'coverage_by_type': coverage['coverage_by_type'],
    },
}

(OUTPUTS / 'run_summary.json').write_text(json.dumps(run_summary, indent=2, default=str))
print('Saved run_summary.json')

print('\nAll outputs:')
for f in sorted(OUTPUTS.iterdir()):
    print(f'  {f.name}')

Saved run_summary.json

All outputs:
  broadband_distribution.png
  choropleth_broadband.html
  choropleth_county_type.html
  choropleth_income.html
  choropleth_lisa.html
  edi_by_cluster.png
  exploratory_correlation.png
  guardrail_analysis.png
  lisa_cluster_map.png
  moran_scatter.png
  regression_comparison.csv
  regression_comparison.png
  report.html
  run_summary.json
  subgroup_regression.png
